1. The "Decay Slope" (Linear Descent)
This is the most direct metric. It calculates the average steepness of the downward slope immediately after the peak.

Why use it: It tells you "The signal dropped X RSAM units per hour."

In [ ]:
import numpy as np
import pandas as pd

def calculate_descent_slope(rsam_data, peak_index, window_size=50):
    """
    Calculates the linear slope of the descent after a peak.
    """
    # Define the window AFTER the peak to analyze
    start = peak_index
    end = min(peak_index + window_size, len(rsam_data) - 1)
    
    if start >= end:
        return np.nan

    # Extract the descent segment
    segment_y = rsam_data[start:end]
    segment_x = np.arange(len(segment_y))
    
    # Fit a simple line (y = mx + c) to get the slope 'm'
    slope, intercept = np.polyfit(segment_x, segment_y, 1)
    
    return slope # A negative number (e.g., -0.5 units/sample)

# Usage example inside your loop:
# slope = calculate_descent_slope(rsam_clean[index], peak_idx)

2. The "Relaxation Time" (Exponential Decay)Volcanic signals often follow a "relaxation" curve where pressure releases quickly at first and then slows down. This is physically more meaningful than a simple straight line.Why use it: It gives you a Decay Constant ($k$) or Half-life. A higher $k$ means the volcano returned to background levels faster.

In [ ]:
from scipy.optimize import curve_fit

def exponential_decay_func(t, A, k, C):
    return A * np.exp(-k * t) + C

def calculate_decay_rate(rsam_data, peak_index, window_size=100):
    """
    Fits an exponential curve to the descent.
    """
    start = peak_index
    end = min(peak_index + window_size, len(rsam_data) - 1)
    
    y_data = rsam_data[start:end]
    # Shift x to start at 0 for easier fitting
    x_data = np.arange(len(y_data))
    
    # Check for empty data
    if len(y_data) < 5: 
        return np.nan

    try:
        # Initial guesses: A=start_height, k=0.1 (slow decay), C=min_height
        p0 = [y_data[0], 0.1, np.min(y_data)]
        
        # Constrain k to be positive (decay)
        bounds = ([-np.inf, 0, -np.inf], [np.inf, np.inf, np.inf])
        
        popt, pcov = curve_fit(exponential_decay_func, x_data, y_data, p0=p0, bounds=bounds, maxfev=1000)
        
        decay_constant = popt[1] # This is 'k'
        return decay_constant
    except:
        return np.nan